In [1]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT,"data","processed")

print("Project Root:")
print(PROJECT_ROOT)

print("\nProcessed Data Directory:")
print(PROCESSED_DATA_DIR)

Project Root:
e:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation

Processed Data Directory:
e:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation\data\processed


In [3]:
print("===== PROCESSED DATA FILES =====")

for file_name in os.listdir(PROCESSED_DATA_DIR):
    print("•", file_name)

===== PROCESSED DATA FILES =====
• clean_houses.geojson
• clean_house_data.csv
• spatial_features.csv


In [4]:
clean_data_path = os.path.join(PROCESSED_DATA_DIR,"clean_house_data.csv")

print("Loading:")
print(clean_data_path)

print("\nFile exists:", os.path.exists(clean_data_path))

Loading:
e:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation\data\processed\clean_house_data.csv

File exists: True


In [5]:
df = pd.read_csv(clean_data_path)

print("Dataset loaded successfully.")

print("\nDataset shape:")
print(df.shape)

display(df.head())

Dataset loaded successfully.

Dataset shape:
(21613, 24)


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,Sale Year,House Age,geometry
0,7129300520,2014-10-13,221900.0,3,1.00,1180,5650,1.0,0,0,...,1955,0,98178,47.5112,-122.257,1340,5650,2014,59,POINT (-122.257 47.5112)
1,6414100192,2014-12-09,538000.0,3,2.25,2570,7242,2.0,0,0,...,1951,1991,98125,47.7210,-122.319,1690,7639,2014,63,POINT (-122.319 47.721)
2,5631500400,2015-02-25,180000.0,2,1.00,770,10000,1.0,0,0,...,1933,0,98028,47.7379,-122.233,2720,8062,2015,82,POINT (-122.233 47.7379)
3,2487200875,2014-12-09,604000.0,4,3.00,1960,5000,1.0,0,0,...,1965,0,98136,47.5208,-122.393,1360,5000,2014,49,POINT (-122.393 47.5208)
4,1954400510,2015-02-18,510000.0,3,2.00,1680,8080,1.0,0,0,...,1987,0,98074,47.6168,-122.045,1800,7503,2015,28,POINT (-122.045 47.6168)


In [6]:
print("===== AVAILABLE FEATURES =====")

for column in df.columns:
    print("•", column)

===== AVAILABLE FEATURES =====
• id
• date
• price
• bedrooms
• bathrooms
• sqft_living
• sqft_lot
• floors
• waterfront
• view
• condition
• grade
• sqft_above
• sqft_basement
• yr_built
• yr_renovated
• zipcode
• lat
• long
• sqft_living15
• sqft_lot15
• Sale Year
• House Age
• geometry


In [7]:
print("===== DATASET INFORMATION =====")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nData Types:")

display(df.dtypes.to_frame(name="Data Type"))

===== DATASET INFORMATION =====
Rows: 21613
Columns: 24

Data Types:


,Data Type
id,int64
date,str
price,float64
bedrooms,int64
bathrooms,float64
sqft_living,int64
sqft_lot,int64
floors,float64
waterfront,int64
view,int64


In [8]:
print("===== MISSING VALUE CHECK =====")

missing_values = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Column": missing_values.index,
    "Missing Values": missing_values.values
})

display(missing_summary)

===== MISSING VALUE CHECK =====


,Column,Missing Values
0,id,0
1,date,0
2,price,0
3,bedrooms,0
4,bathrooms,0
5,sqft_living,0
6,sqft_lot,0
7,floors,0
8,waterfront,0
9,view,0


In [9]:
print("===== DUPLICATE CHECK =====")

duplicate_count = df.duplicated().sum()

print("Duplicate records:", duplicate_count)

if duplicate_count == 0:
    print("✅ No duplicate records found.")
else:
    print("⚠️ Duplicate records detected.")

===== DUPLICATE CHECK =====
Duplicate records: 0
✅ No duplicate records found.


In [10]:
if "yr_built" in df.columns:
    reference_year = int(df["yr_built"].max())
    df["house_age"] = (reference_year - df["yr_built"])
    df["house_age"] = (df["house_age"].clip(lower=0))

    print("House age feature created successfully.")
    print("Reference year:",reference_year)
else:
    print("⚠️ 'yr_built' column not found.")

House age feature created successfully.
Reference year: 2015


In [11]:
if "house_age" in df.columns:
    display(df[["yr_built","house_age"]].head())

,yr_built,house_age
0,1955,60
1,1951,64
2,1933,82
3,1965,50
4,1987,28


In [12]:
if "yr_renovated" in df.columns:
    df["is_renovated"] = (df["yr_renovated"] > 0).astype(int)

    print("Renovation indicator created.")
    print(df["is_renovated"].value_counts())
else:
    print("⚠️ 'yr_renovated' column not found.")

Renovation indicator created.
is_renovated
0    20699
1      914
Name: count, dtype: int64


In [13]:
if ("yr_renovated" in df.columns and "house_age" in df.columns):

    df["years_since_renovation"] = np.where(
        df["yr_renovated"] > 0,
        reference_year - df["yr_renovated"],
        df["house_age"]
    )

    df["years_since_renovation"] = (df["years_since_renovation"].clip(lower=0))

    print("Years-since-renovation feature created.")

    display(df[["yr_renovated","years_since_renovation"]].head())

Years-since-renovation feature created.


,yr_renovated,years_since_renovation
0,0,60
1,1991,24
2,0,82
3,0,50
4,0,28


In [14]:
def haversine_distance(lat1,lon1,lat2,lon2):
    """
    Calculate great-circle distance
    between geographic coordinates.

    Returns distance in kilometers.
    """

    earth_radius_km = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)

    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1

    a = (np.sin(delta_lat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2)

    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius_km * c

In [15]:
SEATTLE_CENTER_LAT = 47.6062
SEATTLE_CENTER_LONG = -122.3321

if ("lat" in df.columns and "long" in df.columns):
    df["distance_to_city_center_km"] = (haversine_distance(df["lat"],df["long"],SEATTLE_CENTER_LAT,SEATTLE_CENTER_LONG))

    print("Distance-to-city-center feature created.")

    display(df[["lat","long","distance_to_city_center_km"]].head())

else:
    print("⚠️ Latitude/longitude columns not found.")

Distance-to-city-center feature created.


,lat,long,distance_to_city_center_km
0,47.5112,-122.257,11.972687
1,47.7210,-122.319,12.802819
2,47.7379,-122.233,16.416960
3,47.5208,-122.393,10.538233
4,47.6168,-122.045,21.553979


In [16]:
if ("sqft_living" in df.columns and "bedrooms" in df.columns):
    df["sqft_per_bedroom"] = (df["sqft_living"] / df["bedrooms"].replace(0, np.nan))

if ("sqft_living" in df.columns and "bathrooms" in df.columns):
    df["sqft_per_bathroom"] = (df["sqft_living"] / df["bathrooms"].replace(0, np.nan))

if ("sqft_living" in df.columns and "sqft_lot" in df.columns):
    df["living_lot_ratio"] = (df["sqft_living"] / df["sqft_lot"].replace(0, np.nan))

print("Additional tabular features created.")

Additional tabular features created.


In [17]:
df = df.replace([np.inf, -np.inf],np.nan)

print("Infinite values handled successfully.")

Infinite values handled successfully.


In [18]:
engineered_features = [
    "house_age",
    "is_renovated",
    "years_since_renovation",
    "distance_to_city_center_km",
    "sqft_per_bedroom",
    "sqft_per_bathroom",
    "living_lot_ratio"
]

available_engineered_features = [column for column in engineered_features if column in df.columns]

print("===== ENGINEERED FEATURES =====")

for column in available_engineered_features:
    print("•", column)

print("\nSummary:")

display(df[available_engineered_features].describe().T)

===== ENGINEERED FEATURES =====
• house_age
• is_renovated
• years_since_renovation
• distance_to_city_center_km
• sqft_per_bedroom
• sqft_per_bathroom
• living_lot_ratio

Summary:


,count,mean,std,min,25%,50%,75%,max
house_age,21613.0,43.994864,29.373411,0.000000,18.000000,40.000000,64.000000,115.000000
is_renovated,21613.0,0.042289,0.201253,0.000000,0.000000,0.000000,0.000000,1.000000
years_since_renovation,21613.0,41.613982,28.806854,0.000000,16.000000,38.000000,61.000000,115.000000
distance_to_city_center_km,21613.0,18.458452,10.641837,0.983119,9.783603,16.509930,25.266094,77.093790
sqft_per_bedroom,21600.0,618.152810,215.884916,49.090909,470.000000,576.666667,722.500000,3420.000000
sqft_per_bathroom,21603.0,1005.599426,293.524116,265.454545,800.000000,970.000000,1165.714286,4600.000000
living_lot_ratio,21613.0,0.323745,0.268565,0.000610,0.156581,0.247664,0.407547,4.653846


In [19]:
print("===== TARGET VARIABLE =====")

if "price" in df.columns:
    print("Target: price")
    print(f"Minimum : ${df['price'].min():,.2f}")
    print(f"Maximum : ${df['price'].max():,.2f}")
    print(f"Mean    : ${df['price'].mean():,.2f}")
    print(f"Median  : ${df['price'].median():,.2f}")

else:
    print("❌ Target column 'price' not found.")

===== TARGET VARIABLE =====
Target: price
Minimum : $75,000.00
Maximum : $1,129,575.00
Mean    : $511,587.28
Median  : $450,000.00


In [20]:
if "price" not in df.columns:
    raise ValueError("Target column 'price' is missing.")

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

feature_columns = [column for column in numeric_columns if column != "price"]

print("Number of numerical features:",len(feature_columns))

print("\nFeatures:")

for column in feature_columns:
    print("•", column)

Number of numerical features: 28

Features:
• id
• bedrooms
• bathrooms
• sqft_living
• sqft_lot
• floors
• waterfront
• view
• condition
• grade
• sqft_above
• sqft_basement
• yr_built
• yr_renovated
• zipcode
• lat
• long
• sqft_living15
• sqft_lot15
• Sale Year
• House Age
• house_age
• is_renovated
• years_since_renovation
• distance_to_city_center_km
• sqft_per_bedroom
• sqft_per_bathroom
• living_lot_ratio


In [21]:
ml_columns = (feature_columns + ["price"])
ml_df = df[ml_columns].copy()

print("ML dataset shape before cleaning:",ml_df.shape)

ML dataset shape before cleaning: (21613, 29)


In [22]:
before_rows = len(ml_df)
ml_df = ml_df.dropna()
after_rows = len(ml_df)

print("Rows before missing-value removal:",before_rows)

print("Rows after missing-value removal :",after_rows)

print("Rows removed:",before_rows - after_rows)

Rows before missing-value removal: 21613
Rows after missing-value removal : 21597
Rows removed: 16


In [23]:
print("===== FINAL ML DATASET VALIDATION =====")

print("Shape:",ml_df.shape)

print("Missing values:",ml_df.isnull().sum().sum())

print("Duplicate rows:",ml_df.duplicated().sum())

display(ml_df.head())

===== FINAL ML DATASET VALIDATION =====
Shape: (21597, 29)
Missing values: 0
Duplicate rows: 4


,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,Sale Year,House Age,house_age,is_renovated,years_since_renovation,distance_to_city_center_km,sqft_per_bedroom,sqft_per_bathroom,living_lot_ratio,price
0,7129300520,3,1.00,1180,5650,1.0,0,0,3,7,...,2014,59,60,0,60,11.972687,393.333333,1180.000000,0.208850,221900.0
1,6414100192,3,2.25,2570,7242,2.0,0,0,3,7,...,2014,63,64,1,24,12.802819,856.666667,1142.222222,0.354874,538000.0
2,5631500400,2,1.00,770,10000,1.0,0,0,3,6,...,2015,82,82,0,82,16.416960,385.000000,770.000000,0.077000,180000.0
3,2487200875,4,3.00,1960,5000,1.0,0,0,5,7,...,2014,49,50,0,50,10.538233,490.000000,653.333333,0.392000,604000.0
4,1954400510,3,2.00,1680,8080,1.0,0,0,3,8,...,2015,28,28,0,28,21.553979,560.000000,840.000000,0.207921,510000.0


In [24]:
feature_path = os.path.join(PROCESSED_DATA_DIR,"tabular_features.csv")

ml_df.to_csv(feature_path,index=False)

print("Week 2 dataset saved successfully.")
print(feature_path)

Week 2 dataset saved successfully.
e:\AARAV\Infotact-DS-ML\Project-3-Geospatial-Valuation\data\processed\tabular_features.csv
